# 03 – Phân cụm + đánh giá (6 thí nghiệm)

Phụ trách: Anh Dũng. Notebook chạy từ thư mục gốc repo hoặc từ `notebooks/`.

Thứ tự: (1) quét số chiều LSA và chọn k → (2) chọn số chiều cho mỗi biểu diễn → (3) bảng 6 thí nghiệm, Elbow/Silhouette → (4) lưu nhãn cụm, cấu hình tốt nhất.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.features.lsa import build_and_save_all_features, load_features
from src.clustering.select_k import MODELS, scan_k, best_k, plot_elbow_silhouette
from src.evaluation.metrics import evaluate, contingency

df = pd.read_csv(config.ARTICLES_CLEAN_CSV)
y = df["true_label"]
LSA_DIMS = [10, 20, 30, 50, 80]
REPS = ["bow_lsa", "tfidf_lsa", "embedding"]

## 1. Quét số chiều LSA, chọn k, chạy thí nghiệm

Với mỗi số chiều LSA (embedding giữ `config.W2V_SIZE` chiều), mỗi biểu diễn và mỗi mô hình: quét k ∈ `config.K_RANGE`, lấy k có Silhouette cao nhất, rồi chạy với k đó và k = 6 (số chủ đề thật).

In [2]:
rows, labels, scans = [], {}, {}
for n in LSA_DIMS:
    build_and_save_all_features(n_components=n)
    for rep in REPS:
        if rep == "embedding" and n != LSA_DIMS[-1]:
            continue
        X = load_features(rep, n_components=n)
        d = X.shape[1]
        for model, run in MODELS.items():
            scans[rep, d, model] = scan_k(X, model)
            for k in sorted({best_k(scans[rep, d, model]), 6}):
                lab = run(X, k)[0]
                labels[rep, d, model, k] = lab
                rows.append({"representation": rep, "n_components": d, "model": model, "k": k,
                             **evaluate(X, lab, y)})

exp = pd.DataFrame(rows).round(4)
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)
exp.to_csv(config.TABLES_DIR / "experiments.csv", index=False)
exp

,representation,n_components,model,k,silhouette,davies_bouldin,calinski_harabasz,ari,nmi,purity
0,bow_lsa,10,kmeans,6,0.4961,1.2978,32.3371,0.5155,0.5943,0.7479
1,bow_lsa,10,kmeans,11,0.5798,0.9768,34.8499,0.4791,0.5918,0.7647
2,bow_lsa,10,kmedoids,6,0.4780,1.3125,30.8168,0.5082,0.5854,0.7059
3,bow_lsa,10,kmedoids,10,0.5850,0.9926,35.5123,0.4936,0.5984,0.7647
4,tfidf_lsa,10,kmeans,6,0.4871,1.1994,30.4549,0.4655,0.5972,0.6807
5,tfidf_lsa,10,kmeans,10,0.6489,0.7850,42.5894,0.4144,0.5831,0.7395
6,tfidf_lsa,10,kmedoids,6,0.4708,1.1661,27.7668,0.4641,0.5722,0.7059
7,tfidf_lsa,10,kmedoids,10,0.6490,0.7711,41.6000,0.3608,0.5601,0.6891
8,bow_lsa,20,kmeans,6,0.3610,1.7288,16.7546,0.4576,0.6259,0.6723
9,bow_lsa,20,kmeans,12,0.4728,1.3620,17.2917,0.3786,0.5846,0.7563


## 2. Chọn số chiều LSA

Chỉ số nội tại (Silhouette, DB, CH) luôn đẹp hơn khi ít chiều nên không dùng để so giữa các số chiều. Chọn số chiều có trung bình ARI và NMI cao nhất ở k = 6, lấy trung bình trên cả hai mô hình.

In [3]:
k6 = exp[exp["k"] == 6].assign(score=lambda t: (t["ari"] + t["nmi"]) / 2)
dims = k6.groupby(["representation", "n_components"])["score"].mean().unstack().round(4)
best_dims = dims.idxmax(axis=1).astype(int).to_dict()
print(best_dims)
dims

{'bow_lsa': 10, 'embedding': 80, 'tfidf_lsa': 10}


n_components,10,20,30,50,80
representation,,,,,
bow_lsa,0.5508,0.5174,0.4964,0.4336,0.4822
embedding,NaN,NaN,NaN,NaN,0.5035
tfidf_lsa,0.5248,0.5142,0.4571,0.4518,0.3988


## 3. Bảng 6 thí nghiệm (số chiều đã chọn) và biểu đồ Elbow / Silhouette

In [4]:
final = exp[[n == best_dims[r] for r, n in zip(exp["representation"], exp["n_components"])]]
final = final.reset_index(drop=True)
final

,representation,n_components,model,k,silhouette,davies_bouldin,calinski_harabasz,ari,nmi,purity
0,bow_lsa,10,kmeans,6,0.4961,1.2978,32.3371,0.5155,0.5943,0.7479
1,bow_lsa,10,kmeans,11,0.5798,0.9768,34.8499,0.4791,0.5918,0.7647
2,bow_lsa,10,kmedoids,6,0.4780,1.3125,30.8168,0.5082,0.5854,0.7059
3,bow_lsa,10,kmedoids,10,0.5850,0.9926,35.5123,0.4936,0.5984,0.7647
4,tfidf_lsa,10,kmeans,6,0.4871,1.1994,30.4549,0.4655,0.5972,0.6807
5,tfidf_lsa,10,kmeans,10,0.6489,0.7850,42.5894,0.4144,0.5831,0.7395
6,tfidf_lsa,10,kmedoids,6,0.4708,1.1661,27.7668,0.4641,0.5722,0.7059
7,tfidf_lsa,10,kmedoids,10,0.6490,0.7711,41.6000,0.3608,0.5601,0.6891
8,embedding,80,kmeans,6,0.2886,1.8865,15.5490,0.4723,0.6048,0.6807
9,embedding,80,kmeans,12,0.3072,1.6580,11.3517,0.4090,0.6004,0.7395


In [5]:
for rep in REPS:
    for model in MODELS:
        plot_elbow_silhouette(scans[rep, best_dims[rep], model],
                              title=f"{rep} ({best_dims[rep]} chiều) – {model}",
                              name=f"elbow_silhouette_{rep}_{model}")

## 4. Lưu nhãn cụm (TV2, TV5 dùng) và cấu hình tốt nhất

In [6]:
out = [pd.DataFrame({"id": df["id"], "representation": r, "n_components": n, "model": m, "k": k,
                     "cluster": labels[r, n, m, k].astype(int)})
       for r, n, m, k in final[["representation", "n_components", "model", "k"]].itertuples(index=False)]
pd.concat(out).to_csv(config.TABLES_DIR / "cluster_labels.csv", index=False)

**Cấu hình tốt nhất: BoW + LSA, K-Means, k = 6.** ARI, NMI và Purity cao nhất trong các thí nghiệm k = 6, Silhouette cao, và mỗi cụm có một chủ đề chiếm đa số nên dễ đặt tên. k chọn theo Silhouette (10–12) cho Silhouette cao hơn nhưng tách thành nhiều cụm nhỏ, ARI thấp hơn. Lý do chi tiết xem `bao_cao_ket_qua/04_phan_cum_danh_gia.md`.

In [7]:
BEST = ("bow_lsa", best_dims["bow_lsa"], "kmeans", 6)
build_and_save_all_features(n_components=BEST[1])
print(final[(final["representation"] == BEST[0]) & (final["model"] == BEST[2]) & (final["k"] == BEST[3])].T)
contingency(y, labels[BEST])

                         0
representation     bow_lsa
n_components            10
model               kmeans
k                        6
silhouette          0.4961
davies_bouldin      1.2978
calinski_harabasz  32.3371
ari                 0.5155
nmi                 0.5943
purity              0.7479


Chủ đề thật,Công nghệ,Giải trí,Kinh tế,Sức khỏe,Thể thao,Thời sự
Cụm,,,,,,
0,12,1,6,0,1,0
1,1,18,2,0,0,0
2,0,0,0,18,1,3
3,0,0,1,0,17,1
4,1,0,9,0,0,1
5,6,1,2,2,0,15
